#### Professor Mounya Elhilali, EN.520.645: Audio Signal Processing
# Project 2: Magnitude Spectrum Inversion
Reconstruct a signal with minimal artifacts from magnitude spectrogram

Deliverable 2: code in Python PART ONE
### Harper Strickland
Due Oct. 8, 2026

#### If any of these libraries are not yet available on your system, use pip install or your preferref method prior to running code

In [1]:
# import libraries
import os
from pydub import AudioSegment
import numpy as np
from numpy.fft import rfft, irfft

#### As written, input files are all mp3 files in a folder called "Audio"

#### The "Audio" folder is located in the same directory as this ipynb file

(change as needed for local setup)

In [2]:
# input mp3 files
load_folder = 'Audio'
files = ['audio1.mp3', 'audio2.mp3', 'audio3.mp3', 'audio4.mp3',
         'audio5.mp3', 'audio6.mp3', 'audio7.mp3', 'audio8.mp3',
         'audio9.mp3', 'audio10.mp3', 'audio11.mp3', 'audio12.mp3', ]
# folder to save can be same or different from folder where original files are
save_folder = 'Results'
# create save folder if doesn't exist
os.makedirs(save_folder, exist_ok=True)

# constant to prevent dividing by zero
reg = 1e-10

##### Create functions to Build RTISI and RTISI-LA

In [3]:
# Hamming window formula
def hamming(m):
    window = 0.54-(0.46 * np.cos((2 * np.pi * np.arange(m))/(m - 1)))
    return window

In [4]:
# function to get array of overlapping frames
# in Fourier domain, windowed (Hamming)
def get_mag_frames(data, win, m, s):
    data = data.astype(np.float64)
    # padding last frame to match
    rmdr = (len(data) - m) % s
    if (rmdr != 0):
        data = np.hstack((data, np.zeros(s - rmdr)))

    frame_count = (len(data) - m) // s + 1
    # create and fill array
    result = np.empty((frame_count, m // 2 + 1))
    for i in range(frame_count):
        frame = data[i * s : i * s + m]
        result[i] = np.abs(rfft(win * frame))

    return result

In [5]:
# step for each interation: test phase estimate/guess value
def test_phase(mframe, guess, prev_out, prev_sum, win, m):
    # test sum of guess and current mag frame
    # inverse transform, 1j = i imaginary
    tframe = irfft(mframe * np.exp(1j * guess), n=m)

    # add to previous, normalize
    test = (prev_out + tframe) / (prev_sum + win + reg)

    # analyze combined estimate, transformed
    spect = rfft(win * test)

    # calculate next guess
    new_guess = np.angle(spect)

    return new_guess

In [6]:
# new frame for look-ahead
def add_frame(k, data, win, out, wsum, guesses, tframes, m, s):
    start = k * s
    end = start + m
    est = out[start:end] / (wsum[start:end] + reg)
    guess = np.angle(rfft(win * est))
    tframe = irfft(data[k] * np.exp(1j * guess), n=m)

    out[start:end] += tframe
    wsum[start:end] += win
    guesses[k] = guess
    tframes[k] = tframe

    return out, wsum, guesses, tframes

In [7]:
# added function for look-ahead functionality
def look_ahead(first, last, data, win, out, wsum, guesses, tframes, m, s):
    # need multiple guesses for all frames in current buffer
    new_guesses = {}
    for k in range(first, last + 1):
        start = k * s
        end = start + m
        # make sure k only added once (reset buffer)
        prev_out = out[start:end] - tframes[k]
        prev_wsum = wsum[start:end] - win
        new_guesses[k] = test_phase(data[k], guesses[k], prev_out, prev_wsum, win, m)

    # apply new guesses
    for k in range(first, last + 1):
        start = k * s
        end = start + m
        # new current frame based on current phase guess
        tframe = irfft(data[k] * np.exp(1j * new_guesses[k]), n=m)
        # update for current iteration
        out[start:end] += tframe - tframes[k]
        tframes[k] = tframe
        guesses[k] = new_guesses[k]

    return out, guesses, tframes

In [8]:
# control iterative reconstruction process
def reconstruct(data, win, m, s, n_iter, l):
    frame_count = data.shape[0]

    # initialize data structures
    out = np.zeros((frame_count - 1) * s + m)
    wsum = out.copy()

    # active buffer frames in look-ahead process
    guesses = {}
    tframes = {}

    for i in range(frame_count):
        # grow the active set: l frames at the start, then one per step
        if i == 0:
            new_frames = range(min(l, frame_count))
        elif i + l - 1 < frame_count:
            new_frames = [i + l - 1]
        else:
            # handles end where no more to look ahead
            new_frames = []

        # pass phase guess to each new frame loop
        for k in new_frames:
            out, wsum, guesses, tframes = add_frame(
                k, data, win, out, wsum, guesses, tframes, m, s)

        last = min(i + l - 1, frame_count - 1)

        # update frames for current buffer
        for j in range(n_iter):
            out, guesses, tframes = look_ahead(
                i, last, data, win, out, wsum, guesses, tframes, m, s)

        # commit frame i
        guesses.pop(i)
        tframes.pop(i)
        
    # normalized by combined window weights
    return out / (wsum + reg)

In [9]:
# signal-to-error ratio (dB)
def ser(mag, orig, win, m, s):
    # reconstructed frames
    new_mag = get_mag_frames(orig, win, m, s)

    # handle mirror for rfft
    w = np.full(mag.shape[1], 2.0)
    w[0] = 1.0
    if m % 2 == 0:
        w[-1] = 1.0

    # numerator: summed energy of target
    num = np.sum(w * mag ** 2)
    # denominator: energy of magnitude error
    den = np.sum(w * (mag - new_mag) ** 2)

    return 10 * np.log10(num / (den + reg))

In [10]:
# full process
def rtisi(file_path, m_ms, s_ms, n_iter, l):
    print('Processing', file_path)
    # load audio
    sound = AudioSegment.from_mp3(file_path)
    rate = sound.frame_rate  # sample rate in Hz
    # frame and step size in samples
    m = int(round(m_ms * rate / 1000))
    s = int(round(s_ms * rate / 1000))
    # setup window for each frame
    win = hamming(m)
    # audio data as numpy array
    raw_data = np.array(sound.get_array_of_samples()).astype('float32')
    chan = sound.channels
    # separate stereo signals from interleaved values
    if (chan == 2):
        # left channel
        magspec = get_mag_frames(raw_data[0::2], win, m, s)
        left = reconstruct(magspec, win, m, s, n_iter, l)
        ser_left = ser(magspec, left, win, m, s)
        # right channel
        magspec = get_mag_frames(raw_data[1::2], win, m, s)
        right = reconstruct(magspec, win, m, s, n_iter, l)
        ser_right = ser(magspec, right, win, m, s)
        print('SER:', round((ser_left + ser_right)/2, 2), 'dB')
        # interleave results
        result = np.column_stack((left, right)).ravel()
    else:
        magspec = get_mag_frames(raw_data, win, m, s)
        result = reconstruct(magspec, win, m, s, n_iter, l)
        print('SER:', round(ser(magspec, result, win, m, s), 2), 'dB')
    # scale to 16-bit
    result = result / np.max(np.abs(result)) * 32767 * 0.9
    result = result.astype(np.int16)
    # result as sound
    out_sound = AudioSegment(data=result.tobytes(), sample_width=2, frame_rate=rate, channels=chan)
    
    return out_sound

### Run RTISI

'l' is set to 1 for RTISI without lookahead (only looking at the current frame and previous for each iteration)
Change any of the other hyperparameter values in the next cell as desired, and run the next two cells as many times as you want.

In [11]:
# hyperparameters
# frame size (ms)
m_ms = 40
# step size (ms)
s_ms = 10
# iterations per frame
n_iter = 10
# SET to 1 to recreate RTISI without Look-Ahead
l = 1
# add to result file name
prefix1 = 'RTISI'

In [12]:
# run RTISI on all given Audio files
for file in files:
    sound_file = load_folder + '/' + file
    final_sound = rtisi(sound_file, m_ms, s_ms, n_iter, l)
    final_sound.export(save_folder + '/' + prefix1 + '_' + file, format='mp3')

Processing Audio/audio1.mp3
SER: 12.64 dB
Processing Audio/audio2.mp3
SER: 10.47 dB
Processing Audio/audio3.mp3
SER: 15.22 dB
Processing Audio/audio4.mp3
SER: 11.21 dB
Processing Audio/audio5.mp3
SER: 15.25 dB
Processing Audio/audio6.mp3
SER: 13.21 dB
Processing Audio/audio7.mp3
SER: 14.14 dB
Processing Audio/audio8.mp3
SER: 18.43 dB
Processing Audio/audio9.mp3
SER: 12.11 dB
Processing Audio/audio10.mp3
SER: 12.63 dB
Processing Audio/audio11.mp3
SER: 13.57 dB
Processing Audio/audio12.mp3
SER: 12.66 dB


### Run RTISI-LA

'l' looks ahead l-1 frames (l = current frame + look-ahead frames)
Change any of the hyperparameter values in the next cell as desired, and run the next two cells as many times as you want.

In [13]:
# hyperparameters
# frame size (ms)
m_ms = 40
# step size (ms)
s_ms = 10
# iterations per frame
n_iter = 5
# frames to look-ahead for RTISI-LA
# includes current frame, minimum = 1
l = 4
# add to result file name
prefix2 = 'RTISI-LA'

In [ ]:
# run RTISI-LA on all given Audio files
for file in files:
    sound_file = load_folder + '/' + file
    final_sound = rtisi(sound_file, m_ms, s_ms, n_iter, l)
    final_sound.export(save_folder + '/' + prefix2 + '_' + file, format='mp3')

Processing Audio/audio1.mp3
SER: 16.25 dB
Processing Audio/audio2.mp3
SER: 13.79 dB
Processing Audio/audio3.mp3
SER: 20.47 dB
Processing Audio/audio4.mp3
SER: 13.22 dB
Processing Audio/audio5.mp3
SER: 19.27 dB
Processing Audio/audio6.mp3
SER: 17.3 dB
Processing Audio/audio7.mp3
SER: 17.31 dB
Processing Audio/audio8.mp3
SER: 23.01 dB
Processing Audio/audio9.mp3
SER: 14.76 dB
Processing Audio/audio10.mp3
